# V5W_09 — Riemann sulla coorte ORIGINALE (#2b: cross-cohort nella stessa geometria)

Risponde alle due cose lasciate aperte: (1) i fenotipi della coorte ORIGINALE emergono anche in geometria di Riemann? (2) OG e V5W si dispongono sullo STESSO asse in uno spazio tangente comune?

Calcola le covarianze SPD della coorte originale (110 parole, P022 escluso) → media di Riemann/soggetto → tangent space → clustering + permutation + ARI con euclideo, poi tangent space congiunto OG+V5W.

**Env: `daniele_311`** (pyriemann sul server). LUNGO: ~90 soggetti × ~550 trial → covarianze (cache).

## §1 — Config

In [ ]:
import json, logging, re
from collections import defaultdict
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from scipy.stats import pearsonr
from tqdm.auto import tqdm
from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace
from pyriemann.utils.mean import mean_riemann

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)-8s %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger('v5w09')
project_root = next((p for p in [Path.cwd()]+list(Path.cwd().parents) if (p/'.git').exists()), Path.cwd())
FIG_DIR=project_root/'figures'; FIG_DIR.mkdir(exist_ok=True)
CKPT=project_root/'models'/'v5w09'; CKPT.mkdir(parents=True, exist_ok=True)
CSV_ROOT=project_root/'data'/'raw_csv'/'training_set'    # coorte ORIGINALE (110 parole)
N_CHAN=61; triu=np.triu_indices(N_CHAN,k=1); SEED=42
EXCLUDE={22}    # P022 outlier (eventualmente aggiungi 23)
_PAT=re.compile(r'^P(\d+)_S(\d+)$')
assert CSV_ROOT.exists(), f'CSV_ROOT non trovato: {CSV_ROOT}'
log.info(f'CSV_ROOT: {CSV_ROOT}  esclusi: {EXCLUDE}')


## §2 — Covarianze SPD per soggetto (coorte originale)

In [ ]:
# Covarianze SPD per soggetto (coorte originale, trial _img). Cache.
CACHE=CKPT/'og_subject_cov.npz'
if CACHE.exists():
    z=np.load(CACHE,allow_pickle=True); M_OG=z['mean']; SUBJ_OG=z['subj'].tolist()
    log.info(f'Cov OG da cache: {M_OG.shape}')
else:
    cov_est=Covariances(estimator='oas')
    subj_dirs=defaultdict(list)
    for d in sorted(CSV_ROOT.iterdir()):
        m=_PAT.match(d.name)
        if m and int(m.group(1)) not in EXCLUDE: subj_dirs[int(m.group(1))].append(d)
    SUBJ_OG=[]; means=[]
    for sid in tqdm(sorted(subj_dirs), desc='cov/soggetto OG'):
        X=[]
        for sd in subj_dirs[sid]:
            for csv in sorted(sd.glob('*_img.csv')):
                if csv.name.startswith('._'): continue
                a=pd.read_csv(csv,header=None).values.astype(np.float32)
                if a.shape==(N_CHAN,384): X.append(a)
        if len(X)<10: continue
        C=cov_est.transform(np.stack(X))           # (n,61,61) SPD
        means.append(mean_riemann(C)); SUBJ_OG.append(sid)
    M_OG=np.stack(means); np.savez(CACHE, mean=M_OG, subj=np.array(SUBJ_OG))
    log.info(f'Cov OG calcolate: {M_OG.shape} ({len(SUBJ_OG)} soggetti)')
print('Soggetti OG:', len(SUBJ_OG))


## §3 — Clustering Riemann + permutation + ARI vs euclideo

In [ ]:
# Clustering Riemann sulla coorte ORIGINALE + permutation + ARI con euclideo
def det1(C):
    s,ld=np.linalg.slogdet(C); return C*np.exp(-ld/C.shape[0])
Mn=np.stack([det1(M_OG[i]) for i in range(len(M_OG))])   # scala rimossa
TS=TangentSpace().fit(Mn).transform(Mn)                   # tangent space

def sil_k2(feat):
    Z=PCA(n_components=min(20,feat.shape[0]-1),random_state=SEED).fit_transform(StandardScaler().fit_transform(feat))
    lab=KMeans(2,n_init=20,random_state=SEED).fit_predict(Z); return silhouette_score(Z,lab),lab
def perm(feat,nperm=1000,seed=SEED):
    obs,_=sil_k2(feat); rng=np.random.default_rng(seed); n,d=feat.shape; null=np.empty(nperm)
    for i in range(nperm):
        idx=rng.random((n,d)).argsort(0); null[i],_=sil_k2(np.take_along_axis(feat,idx,0))
    return obs,null,(null>=obs).mean()

obs,null,p=perm(TS); _,LAB_R=sil_k2(TS)
# euclideo sulle stesse covarianze (triu del mean cov)
FEAT_E=np.array([M_OG[i][triu] for i in range(len(M_OG))])
_,LAB_E=sil_k2(FEAT_E); ari=adjusted_rand_score(LAB_R,LAB_E)
print('='*60)
print('RIEMANN sulla coorte ORIGINALE')
print(f'  silhouette k=2 (Riemann, det=1) = {obs:.3f}  null={null.mean():.3f}  p={p:.3f}')
print(f'  ARI(Riemann, euclideo) sulle stesse matrici = {ari:.3f}')
print('='*60)
np.savez(CKPT/'og_riemann_labels.npz', subj=np.array(SUBJ_OG), label=LAB_R)


## §4 — Cross-cohort nello spazio tangente comune (OG + V5W)

In [ ]:
# Cross-cohort NELLA STESSA GEOMETRIA: tangent space congiunto OG + V5W
v5w_cache=project_root/'models'/'v5w07'/'covs.npz'
if not v5w_cache.exists():
    print('Cache V5W covs.npz assente (esegui V5W_07 §2). Salto il cross-cohort congiunto.')
else:
    z=np.load(v5w_cache,allow_pickle=True); COVS_V=z['covs']; SUBJ_V=z['subj']
    means_v=[]; subj_v=[]
    for sid in sorted(set(SUBJ_V.tolist())):
        Cs=COVS_V[SUBJ_V==sid]
        if len(Cs)>=10: means_v.append(mean_riemann(Cs)); subj_v.append(sid)
    M_V=np.stack(means_v)
    # tangent space COMUNE (media di Riemann su OG+V5W insieme)
    allM=np.concatenate([M_OG,M_V],axis=0)
    allM_n=np.stack([det1(C) for C in allM])
    ts=TangentSpace().fit(allM_n); T=ts.transform(allM_n)
    Z=PCA(5,random_state=SEED).fit_transform(StandardScaler().fit_transform(T))
    nog=len(M_OG)
    # asse fenotipico in OG e in V5W nello spazio comune (diff medie cluster su PC)
    lab_og=LAB_R; 
    _,lab_v=sil_k2(TangentSpace().fit(np.stack([det1(c) for c in M_V])).transform(np.stack([det1(c) for c in M_V])))
    pc1=Z[:,0]
    og_sep=pc1[:nog][lab_og==1].mean()-pc1[:nog][lab_og==0].mean()
    v_sep =pc1[nog:][lab_v==1].mean()-pc1[nog:][lab_v==0].mean()
    print('='*60)
    print('CROSS-COHORT nello SPAZIO TANGENTE COMUNE (OG+V5W)')
    print(f'  separazione fenotipi su PC1 comune: OG={og_sep:+.2f}  V5W={v_sep:+.2f}')
    print(f'  stesso segno = i due fenotipi si dispongono sullo STESSO asse comune: {"SI" if og_sep*v_sep>0 else "NO"}')
    print('='*60)
    fig,ax=plt.subplots(figsize=(8,5))
    ax.scatter(Z[:nog,0],Z[:nog,1],c=['#2166AC' if l==0 else '#D6604D' for l in lab_og],marker='o',s=55,edgecolor='k',lw=0.4,label='OG')
    ax.scatter(Z[nog:,0],Z[nog:,1],c=['#2166AC' if l==0 else '#D6604D' for l in lab_v],marker='^',s=70,edgecolor='k',lw=0.4,label='V5W')
    ax.set_xlabel('Tangent PC1 (comune)'); ax.set_ylabel('PC2'); ax.legend()
    ax.set_title('OG (cerchi) + V5W (triangoli) nello stesso spazio di Riemann\ncolore = fenotipo (blu C0 / rosso C1)',fontweight='bold')
    plt.tight_layout(); plt.savefig(FIG_DIR/'v5w09_joint_tangent.png',dpi=160,bbox_inches='tight'); plt.show()
